In [1]:

import pandas as pd
import numpy as np

from itertools import combinations

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    StratifiedKFold,
    RandomizedSearchCV
)
# ============================================================
# SECTION 1: IMPORT FEATURE SELECTION
# ============================================================

from sklearn.feature_selection import RFECV
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectFromModel


In [2]:

# Import dataset
df = pd.read_csv("ACME-HappinessSurvey2020.csv")

# Separate predictors and target
X = df.drop(columns=["Y"])
y = df["Y"].astype(int)

# Generate all 15 pairwise interactions
all_interactions = list(combinations(X.columns, 2))

def add_interactions(data, interactions):

    result = data.copy()

    for a, b in interactions:
        result[f"{a}_x_{b}"] = data[a] * data[b]

    return result

# Generate features using the entire dataset
X_rf = add_interactions(X, all_interactions)

print("Dataset dimensions:", X_rf.shape)
print("Number of observations:", len(y))
print("Number of predictors:", X_rf.shape[1])

Dataset dimensions: (126, 21)
Number of observations: 126
Number of predictors: 21


In [3]:
# ============================================================
# SECTION 1: GENERATE THE 18 AGGREGATE FEATURES
# ============================================================

question_cols = [f"X{i}" for i in range(1, 7)]

X_original = df[question_cols].copy()

# Start with an empty feature dataframe
X_rf = pd.DataFrame(index=df.index)

# Generate aggregate features excluding each individual question
for feature in question_cols:

    other_features = [
        col for col in question_cols
        if col != feature
    ]

    # Count of responses equal to 5
    X_rf[f"{feature}_other_count5"] = (
        X_original[other_features] == 5
    ).sum(axis=1)

    # Count of responses >= 4
    X_rf[f"{feature}_other_count4plus"] = (
        X_original[other_features] >= 4
    ).sum(axis=1)

    # Count of responses <= 2
    X_rf[f"{feature}_other_count2minus"] = (
        X_original[other_features] <= 2
    ).sum(axis=1)


# ============================================================
# SECTION 2: DEFINE TARGET
# ============================================================

y = df["Y"].astype(int)


# ============================================================
# SECTION 3: INSPECT FEATURES
# ============================================================

print("Observations:", len(X_rf))
print("Number of predictors:", X_rf.shape[1])

print("\nFeatures:")
print(X_rf.columns.tolist())

print("\nFirst five observations:")
print(X_rf.head())

Observations: 126
Number of predictors: 18

Features:
['X1_other_count5', 'X1_other_count4plus', 'X1_other_count2minus', 'X2_other_count5', 'X2_other_count4plus', 'X2_other_count2minus', 'X3_other_count5', 'X3_other_count4plus', 'X3_other_count2minus', 'X4_other_count5', 'X4_other_count4plus', 'X4_other_count2minus', 'X5_other_count5', 'X5_other_count4plus', 'X5_other_count2minus', 'X6_other_count5', 'X6_other_count4plus', 'X6_other_count2minus']

First five observations:
   X1_other_count5  X1_other_count4plus  X1_other_count2minus  \
0                0                    2                     1   
1                1                    2                     1   
2                1                    1                     0   
3                1                    2                     0   
4                1                    2                     0   

   X2_other_count5  X2_other_count4plus  X2_other_count2minus  \
0                0                    2                     1   
1 

In [4]:

# ============================================================
# SECTION 2: DEFINE HYPERPARAMETER SEARCH
# ============================================================
n_splits=7
param_grid = {
    'n_estimators': [30,40,50,60,70],
    'max_depth': [None, 3, 5, 8, 12],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.5, 1.0],
    'class_weight': [None, 'balanced']
}


# ============================================================
# SECTION 3: NESTED CROSS-VALIDATION WITH RFECV
# ============================================================

outer_cv = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=42
)

outer_accuracies = []
oof_predictions = np.empty(len(y), dtype=int)

selected_features_per_fold = []

for fold, (train_idx, val_idx) in enumerate(
    outer_cv.split(X_rf, y), start=1
):

    print(f"\nPROCESSING FOLD {fold}")
    print("=" * 50)

    # Outer training and validation data
    X_outer_train = X_rf.iloc[train_idx]
    X_outer_val = X_rf.iloc[val_idx]

    y_outer_train = y.iloc[train_idx]
    y_outer_val = y.iloc[val_idx]

    # Inner cross-validation
    inner_cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )


    # --------------------------------------------------------
    # STEP 1: FEATURE SELECTION USING RFECV
    # --------------------------------------------------------

    rf_selector = RandomForestClassifier(
        n_estimators=30,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=1
    )

    selector = RFECV(
        estimator=rf_selector,
        step=1,
        min_features_to_select=1,
        cv=inner_cv,
        scoring='accuracy',
        n_jobs=-1
    )

    # Select features using outer training data only
    selector.fit(
        X_outer_train,
        y_outer_train
    )

    # Obtain selected feature names
    selected_features = X_outer_train.columns[
        selector.support_
    ].tolist()

    selected_features_per_fold.append(
        selected_features
    )

    print("Original Features:", X_rf.shape[1])
    print("Selected Features:", len(selected_features))
    print("Retained Features:", selected_features)


    # --------------------------------------------------------
    # STEP 2: HYPERPARAMETER TUNING
    # --------------------------------------------------------

    X_train_selected = X_outer_train[selected_features]
    X_val_selected = X_outer_val[selected_features]

    rf = RandomForestClassifier(
        random_state=42,
        n_jobs=1
    )

    rf_search = RandomizedSearchCV(
        estimator=rf,
        param_distributions=param_grid,
        n_iter=20,
        scoring='accuracy',
        cv=inner_cv,
        random_state=42,
        n_jobs=-1,
        refit=True
    )

    rf_search.fit(
        X_train_selected,
        y_outer_train
    )


    # --------------------------------------------------------
    # STEP 3: EVALUATE OUTER VALIDATION FOLD
    # --------------------------------------------------------

    best_rf = rf_search.best_estimator_

    y_pred = best_rf.predict(X_val_selected)

    fold_accuracy = accuracy_score(
        y_outer_val,
        y_pred
    )

    outer_accuracies.append(fold_accuracy)

    oof_predictions[val_idx] = y_pred

    print("Best Parameters:", rf_search.best_params_)
    print("Outer Fold Accuracy:", round(fold_accuracy, 4))


# ============================================================
# SECTION 4: DISPLAY CROSS-VALIDATION RESULTS
# ============================================================

print("\nNESTED CROSS-VALIDATION RESULTS")
print("=" * 50)

print("Fold Accuracies:", np.round(outer_accuracies, 4))

print(
    "Mean Accuracy:",
    round(np.mean(outer_accuracies), 4)
)

print(
    "Standard Deviation:",
    round(np.std(outer_accuracies, ddof=1), 4)
)

print(
    "Overall OOF Accuracy:",
    round(accuracy_score(y, oof_predictions), 4)
)


PROCESSING FOLD 1
Original Features: 18
Selected Features: 16
Retained Features: ['X1_other_count5', 'X1_other_count4plus', 'X2_other_count5', 'X2_other_count4plus', 'X2_other_count2minus', 'X3_other_count5', 'X3_other_count4plus', 'X4_other_count5', 'X4_other_count4plus', 'X4_other_count2minus', 'X5_other_count5', 'X5_other_count4plus', 'X5_other_count2minus', 'X6_other_count5', 'X6_other_count4plus', 'X6_other_count2minus']
Best Parameters: {'n_estimators': 30, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 5, 'class_weight': 'balanced'}
Outer Fold Accuracy: 0.7778

PROCESSING FOLD 2
Original Features: 18
Selected Features: 2
Retained Features: ['X4_other_count5', 'X5_other_count4plus']
Best Parameters: {'n_estimators': 70, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': None, 'class_weight': 'balanced'}
Outer Fold Accuracy: 0.5

PROCESSING FOLD 3
Original Features: 18
Selected Features: 2
Retained Features: [

In [5]:
# ============================================================
# SECTION 5: FEATURE SELECTION FREQUENCY
# ============================================================

feature_frequency = pd.Series(
    {
        feature: sum(
            feature in selected
            for selected in selected_features_per_fold
        )
        for feature in X_rf.columns
    }
).sort_values(ascending=False)

n_folds = len(selected_features_per_fold)

frequency_df = pd.DataFrame({
    "Feature": feature_frequency.index,
    "Selected Folds": feature_frequency.values,
    "Selection Frequency (%)":
        feature_frequency.values / n_folds * 100
})

print("\nFEATURE SELECTION FREQUENCY")
print("=" * 50)

print(frequency_df.to_string(index=False))


FEATURE SELECTION FREQUENCY
             Feature  Selected Folds  Selection Frequency (%)
     X4_other_count5               7               100.000000
 X6_other_count4plus               6                85.714286
 X5_other_count4plus               6                85.714286
 X2_other_count4plus               5                71.428571
 X3_other_count4plus               4                57.142857
     X2_other_count5               4                57.142857
 X4_other_count4plus               3                42.857143
     X1_other_count5               3                42.857143
X3_other_count2minus               2                28.571429
 X1_other_count4plus               2                28.571429
     X3_other_count5               2                28.571429
X2_other_count2minus               2                28.571429
     X6_other_count5               2                28.571429
X6_other_count2minus               2                28.571429
X4_other_count2minus               1     

In [6]:
# ============================================================
# SECTION 6: FINAL FEATURE SELECTION ON ALL DATA
# ============================================================

final_selector = RFECV(
    estimator=RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=1
    ),
    step=1,
    min_features_to_select=1,
    cv=StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    ),
    scoring='accuracy',
    n_jobs=-1
)

final_selector.fit(X_rf, y)

final_features = X_rf.columns[
    final_selector.support_
].tolist()

X_final = X_rf[final_features]

print("Original number of features:", X_rf.shape[1])
print("Selected number of features:", len(final_features))

print("\nFinal Selected Features:")
print(final_features)


# ============================================================
# SECTION 7: FINAL RANDOM FOREST TUNING
# ============================================================

final_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(
        random_state=42,
        n_jobs=1
    ),
    param_distributions=param_grid,
    n_iter=30,
    scoring='accuracy',
    cv=StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    ),
    random_state=42,
    n_jobs=-1,
    refit=True
)

final_search.fit(X_final, y)

best_rf_model = final_search.best_estimator_

print("\nFinal Best Parameters:")
print(final_search.best_params_)

print("\nFinal Tuning CV Accuracy:")
print(final_search.best_score_)

Original number of features: 18
Selected number of features: 4

Final Selected Features:
['X2_other_count4plus', 'X4_other_count5', 'X5_other_count4plus', 'X6_other_count4plus']

Final Best Parameters:
{'n_estimators': 30, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 3, 'class_weight': 'balanced'}

Final Tuning CV Accuracy:
0.7301587301587301


In [7]:
# ============================================================
# SECTION 8: TRAIN FINAL MODEL USING ALL AVAILABLE DATA
# ============================================================

# Extract the best hyperparameters
best_params = final_search.best_params_

print("Selected Hyperparameters:")
print(best_params)

# Create final Random Forest with the selected hyperparameters
final_rf_model = RandomForestClassifier(
    **best_params,
    random_state=42,
    n_jobs=-1
)

# Train using ALL observations and selected features
final_rf_model.fit(X_final, y)

# Confirm final model
print("\nFINAL RANDOM FOREST MODEL")
print("=" * 50)

print(final_rf_model)

print("\nNumber of training observations:", len(y))
print("Number of features used:", X_final.shape[1])

print("\nFeatures used:")
print(final_features)

Selected Hyperparameters:
{'n_estimators': 30, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 3, 'class_weight': 'balanced'}

FINAL RANDOM FOREST MODEL
RandomForestClassifier(class_weight='balanced', max_depth=3,
                       max_features='log2', min_samples_split=10,
                       n_estimators=30, n_jobs=-1, random_state=42)

Number of training observations: 126
Number of features used: 4

Features used:
['X2_other_count4plus', 'X4_other_count5', 'X5_other_count4plus', 'X6_other_count4plus']


In [8]:
# ============================================================
# SECTION 9: FINAL MODEL TRAINING EVALUATION
# ============================================================

# Generate predictions on all training observations
y_train_pred = final_rf_model.predict(X_final)

# Calculate training accuracy
train_accuracy = accuracy_score(y, y_train_pred)

print("\nFINAL MODEL - TRAINING EVALUATION")
print("=" * 50)

print("Number of observations:", len(y))
print("Training Accuracy:", round(train_accuracy, 4))
print("Training Accuracy (%):", round(train_accuracy * 100, 2))


FINAL MODEL - TRAINING EVALUATION
Number of observations: 126
Training Accuracy: 0.7381
Training Accuracy (%): 73.81
